# Transfer Learning for Pneumonia Detection

This notebook implements transfer learning using pre-trained CNN models (VGG16, ResNet50, and others) for pneumonia detection.

In [ ]:
import sys
import os
import numpy as np
import tensorflow as tf
from tensorflow import keras

sys.path.append(os.path.join(os.path.dirname(os.getcwd()), 'src'))

from src.models.transfer_learning import (
    build_vgg16_model, build_resnet50_model, 
    build_inceptionv3_model, build_efficientnet_model, build_densenet_model
)
from src.models.train import train_model
from src.models.evaluate import evaluate_model, get_confusion_matrix, get_classification_report
from src.utils.visualization import plot_training_history, plot_confusion_matrix, plot_roc_curve
from src.utils.config import CLASS_LABELS, BEST_MODEL_PATH, IMG_SIZE, NUM_CLASSES

# Set random seeds
np.random.seed(42)
tf.random.set_seed(42)

## 1. Load Preprocessed Data

Note: Ensure preprocessed data is available from previous notebooks. For transfer learning, we need RGB images (3 channels).

In [ ]:
# Load preprocessed data
# Note: For transfer learning, convert grayscale to RGB by repeating channels
# Uncomment if you saved preprocessed data:
# import pickle
# with open('data/processed/preprocessed_data.pkl', 'rb') as f:
#     data = pickle.load(f)
#     X_train = data['X_train']
#     X_val = data['X_val']
#     X_test = data['X_test']
#     y_train = data['y_train']
#     y_val = data['y_val']
#     y_test = data['y_test']

# Convert grayscale to RGB for transfer learning models
# X_train_rgb = np.repeat(X_train, 3, axis=-1)
# X_val_rgb = np.repeat(X_val, 3, axis=-1)
# X_test_rgb = np.repeat(X_test, 3, axis=-1)

print("Please ensure preprocessed data is loaded and converted to RGB format")

## 2. Build and Train VGG16 Model

In [ ]:
# Build VGG16 model
model_vgg = build_vgg16_model(
    input_shape=(*IMG_SIZE, 3),
    num_classes=NUM_CLASSES,
    freeze_base=True
)

model_vgg.summary()

In [ ]:
# Train VGG16 model
history_vgg = train_model(
    model_vgg,
    X_train_rgb, y_train,
    X_val_rgb, y_val,
    epochs=30,
    batch_size=32,
    use_class_weights=True,
    use_augmentation=True,
    model_save_path='models/saved_models/best_model_vgg16.h5',
    logs_dir='logs/training_logs'
)

plot_training_history(history_vgg)

In [ ]:
# Evaluate VGG16
best_model_vgg = keras.models.load_model('models/saved_models/best_model_vgg16.h5')
metrics_vgg = evaluate_model(best_model_vgg, X_test_rgb, y_test)

print("VGG16 Test Performance:")
for key, value in metrics_vgg.items():
    print(f"{key}: {value:.4f}")

## 3. Build and Train ResNet50 Model

In [ ]:
# Build ResNet50 model
model_resnet = build_resnet50_model(
    input_shape=(*IMG_SIZE, 3),
    num_classes=NUM_CLASSES,
    freeze_base=True
)

model_resnet.summary()

In [ ]:
# Train ResNet50 model
history_resnet = train_model(
    model_resnet,
    X_train_rgb, y_train,
    X_val_rgb, y_val,
    epochs=30,
    batch_size=32,
    use_class_weights=True,
    use_augmentation=True,
    model_save_path='models/saved_models/best_model_resnet50.h5',
    logs_dir='logs/training_logs'
)

plot_training_history(history_resnet)

In [ ]:
# Evaluate ResNet50
best_model_resnet = keras.models.load_model('models/saved_models/best_model_resnet50.h5')
metrics_resnet = evaluate_model(best_model_resnet, X_test_rgb, y_test)

print("ResNet50 Test Performance:")
for key, value in metrics_resnet.items():
    print(f"{key}: {value:.4f}")

## 4. Create New Architectures with Additional Layers

Let's create enhanced versions by adding more custom layers.

In [ ]:
# Enhanced VGG16 with additional layers
from tensorflow.keras.applications import VGG16
from tensorflow.keras import layers, Model

base_vgg = VGG16(weights='imagenet', include_top=False, input_shape=(*IMG_SIZE, 3))
base_vgg.trainable = False

inputs = keras.Input(shape=(*IMG_SIZE, 3))
x = base_vgg(inputs, training=False)
x = layers.GlobalAveragePooling2D()(x)
x = layers.Dense(1024, activation='relu')(x)
x = layers.BatchNormalization()(x)
x = layers.Dropout(0.5)(x)
x = layers.Dense(512, activation='relu')(x)
x = layers.BatchNormalization()(x)
x = layers.Dropout(0.5)(x)
x = layers.Dense(256, activation='relu')(x)
outputs = layers.Dense(NUM_CLASSES, activation='softmax')(x)

model_vgg_enhanced = Model(inputs, outputs)
model_vgg_enhanced.compile(
    optimizer=keras.optimizers.Adam(learning_rate=0.001),
    loss='sparse_categorical_crossentropy',
    metrics=['accuracy']
)

model_vgg_enhanced.summary()

In [ ]:
# Train enhanced VGG16
history_vgg_enhanced = train_model(
    model_vgg_enhanced,
    X_train_rgb, y_train,
    X_val_rgb, y_val,
    epochs=30,
    batch_size=32,
    use_class_weights=True,
    use_augmentation=True,
    model_save_path='models/saved_models/best_model_vgg16_enhanced.h5',
    logs_dir='logs/training_logs'
)

plot_training_history(history_vgg_enhanced)

## 5. Compare All Models

In [ ]:
# Compare all models
import pandas as pd

# Load custom CNN results (from previous notebook)
# metrics_cnn = {...}  # Fill from CNN notebook

results = {
    'Model': ['Custom CNN', 'VGG16', 'ResNet50', 'VGG16 Enhanced'],
    'Accuracy': [0.0, metrics_vgg['accuracy'], metrics_resnet['accuracy'], 0.0],  # Update with actual values
    'Precision': [0.0, metrics_vgg['precision'], metrics_resnet['precision'], 0.0],
    'Recall': [0.0, metrics_vgg['recall'], metrics_resnet['recall'], 0.0],
    'F1-Score': [0.0, metrics_vgg['f1_score'], metrics_resnet['f1_score'], 0.0],
    'ROC-AUC': [0.0, metrics_vgg['roc_auc'], metrics_resnet['roc_auc'], 0.0]
}

results_df = pd.DataFrame(results)
print(results_df.to_string(index=False))

# Visualize comparison
import matplotlib.pyplot as plt
results_df.set_index('Model').plot(kind='bar', figsize=(12, 6))
plt.title('Model Performance Comparison')
plt.ylabel('Score')
plt.xticks(rotation=45, ha='right')
plt.legend(loc='upper left')
plt.tight_layout()
plt.show()

## 6. Choose Best Model and Serialize

Based on the comparison, select the best performing model.

In [ ]:
# Choose best model based on evaluation
# Example: If ResNet50 performs best
best_model = keras.models.load_model('models/saved_models/best_model_resnet50.h5')

# Save as the final best model
best_model.save(BEST_MODEL_PATH)
print(f"Best model saved to {BEST_MODEL_PATH}")

# Verify model can be loaded
loaded_model = keras.models.load_model(BEST_MODEL_PATH)
print("Model successfully serialized and loaded!")

## 7. Make Inferences on Sample Images

In [ ]:
# Make predictions on a few test images
from src.models.predict import predict_pneumonia
import matplotlib.pyplot as plt

# Select a few test images
num_samples = 5
sample_indices = np.random.choice(len(X_test_rgb), num_samples, replace=False)

fig, axes = plt.subplots(1, num_samples, figsize=(15, 3))
for i, idx in enumerate(sample_indices):
    image = X_test_rgb[idx]
    true_label = y_test[idx]
    
    # Make prediction (convert back to single channel for prediction function)
    # Note: predict_pneumonia expects single channel, so we'll use model directly
    pred_proba = loaded_model.predict(np.expand_dims(image, axis=0), verbose=0)
    pred_class = np.argmax(pred_proba[0])
    confidence = pred_proba[0][pred_class]
    
    # Display image
    axes[i].imshow(image[:, :, 0], cmap='gray')
    axes[i].set_title(f'True: {CLASS_LABELS[true_label]}\\nPred: {CLASS_LABELS[pred_class]}\\nConf: {confidence:.2f}')
    axes[i].axis('off')

plt.tight_layout()
plt.show()

## 8. Model Comparison and Rationale

### Performance Summary:
- **Custom CNN**: [Fill with results]
- **VGG16**: [Fill with results]
- **ResNet50**: [Fill with results]
- **VGG16 Enhanced**: [Fill with results]

### Best Model Selection:
**Selected Model**: [Model Name]

**Rationale**:
1. [Reason 1 - e.g., Highest accuracy]
2. [Reason 2 - e.g., Best F1-score for imbalanced data]
3. [Reason 3 - e.g., Good balance between performance and inference time]
4. [Reason 4 - e.g., Generalization to test set]

### Key Observations:
- Transfer learning models generally perform better than custom CNN
- [Additional observations]
- [Recommendations for production deployment]